In [2]:
import asyncio
from playwright.async_api import async_playwright
import pandas as pd
import nest_asyncio

nest_asyncio.apply()

class YandexWordstatParser:
    def __init__(self):
        self.results = []

    async def parse_wordstat(self, keyword: str) -> list[dict]:
        self.results = []
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=False)
            page = await browser.new_page()
            
            await page.goto('https://wordstat.yandex.ru')
            
            input("После входа нажмите Enter...")
            
            await page.wait_for_timeout(3000)
            
            search_input = await page.query_selector('input[name="text"]')
            if search_input is None:
                search_input = await page.query_selector('input[type="search"]')
            if search_input is None:
                search_input = await page.query_selector('input')
            
            if search_input:
                await search_input.fill(keyword)
                await search_input.press('Enter')
                
                await page.wait_for_timeout(8000)
                
                rows = await page.query_selector_all('tr')
                
                for row in rows[:25]:
                    cells = await row.query_selector_all('td')
                    if len(cells) >= 2:
                        phrase = await cells[0].inner_text()
                        volume = await cells[1].inner_text()
                        
                        if phrase and volume and phrase.strip():
                            self.results.append({
                                'исходный_запрос': keyword,
                                'связанный_запрос': phrase.strip(),
                                'показов_в_месяц': volume.strip().replace(' ', '')
                            })
            
            await browser.close()
        return self.results

    async def parse_multiple_keywords(self, keywords: list[str]) -> list[dict]:
        all_results = []
        for keyword in keywords:
            data = await self.parse_wordstat(keyword)
            all_results.extend(data)
        return all_results


async def main():
    parser = YandexWordstatParser()
    keywords = ['бизнес калькулятор']
    results = await parser.parse_multiple_keywords(keywords)
    
    if results:
        df = pd.DataFrame(results)
        print(df.to_string(index=False))
        df.to_csv('wordstat_data.csv', index=False, encoding='utf-8-sig')

await main()

После входа нажмите Enter...
   исходный_запрос                            связанный_запрос показов_в_месяц
бизнес калькулятор                           формула стоимости           94250
бизнес калькулятор                                 расчет веса           46580
бизнес калькулятор                        расчет себестоимости           21618
бизнес калькулятор                                 расчет кбжу           19017
бизнес калькулятор                    деловые линии рассчитать           16697
бизнес калькулятор          деловые линии рассчитать стоимость           11448
бизнес калькулятор деловые линии рассчитать стоимость доставки            9599
бизнес калькулятор                       расчет рентабельности            8280
бизнес калькулятор                                  расчет бжу            6325
бизнес калькулятор                   калькуляция себестоимости            6194
бизнес калькулятор                как посчитать рентабельность            3617
бизнес калькулятор     